# AnchorKV: FP16 arithmetic development gate

Fresh Colab T4, Run all. **Only 12 FP16 generations**, no compression sweep,
Triton compilation, throughput benchmark or held-out model evaluation.
This checks readiness; the baseline is NOT already validated.
Six underlying arithmetic problems each have a clean and distractor context.
Cap: 64 generated tokens. Soft phase limit: 8 minutes after loading weights.
Checkpoints resume by rerunning the generation cell in the same live runtime.
Downloads/setup and in-flight generation are outside the soft limit.
Preserve the output folder on Drive if you need to survive runtime loss.
Return the final zip whether the gate passes or fails; do not drop hard cases.

In [ ]:
import subprocess
import sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'transformers==4.57.6', 'huggingface_hub>=0.34,<1', 'accelerate>=1,<2'])

In [ ]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import platform
import shutil
import time
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
assert transformers.__version__ == '4.57.6', 'Restart the runtime after installing.'
assert torch.cuda.is_available(), 'Select a T4 GPU runtime.'
SOURCES = {'__init__.py': '', 'arithmetic_protocol.py': '"""Deterministic development/held-out arithmetic protocol. No model or GPU needed."""\nimport argparse\nfrom collections import Counter\nimport hashlib\nimport json\nfrom pathlib import Path\nimport random\n\nfrom .arithmetic_scoring import SCORER_VERSION, score_response\n\nPROTOCOL_VERSION = \'arithmetic-protocol-v1\'\nMODEL_ID = \'Qwen/Qwen3-0.6B\'\nMODEL_REVISION = \'c1899de289a04d12100db370d81485cdf75e47ca\'\nOPERATIONS = (\'addition\', \'subtraction\', \'multiply_subtract\')\nSYSTEM = (\'Solve the requested inventory calculation using only the Zephyr records. \'\n          \'Other projects are unrelated. Return only the final integer, without units or explanation.\')\n\n\ndef canonical_hash(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=True).encode()).hexdigest()\n\n\ndef build_protocol():\n    splits = {\'development\': [], \'heldout\': []}\n    used = set()\n    for split, per_operation in ((\'development\', 2), (\'heldout\', 4)):\n        for operation in OPERATIONS:\n            for index in range(per_operation):\n                family = f\'{split}-{operation}-{index:02}\'\n                rng = random.Random(f\'{PROTOCOL_VERSION}:{family}\')\n                while True:\n                    a, b, c = rng.randint(40, 90), rng.randint(2, 30), 0\n                    if operation == \'multiply_subtract\':\n                        a, b, c = rng.randint(3, 12), rng.randint(4, 15), rng.randint(1, 9)\n                    key = operation, a, b, c\n                    if key not in used:\n                        used.add(key)\n                        break\n                if operation == \'addition\':\n                    facts = [f\'Zephyr opening inventory: {a} units.\', f\'Zephyr received: {b} units.\']\n                    query = \'What is Zephyr inventory after receiving the delivery?\'\n                    gold = a + b\n                elif operation == \'subtraction\':\n                    facts = [f\'Zephyr opening inventory: {a} units.\', f\'Zephyr shipped: {b} units.\']\n                    query = \'What is Zephyr inventory after shipping?\'\n                    gold = a - b\n                else:\n                    facts = [f\'Zephyr received: {a} crates.\', f\'Zephyr units per crate: {b}.\',\n                             f\'Zephyr shipped after receiving: {c} units.\']\n                    query = \'How many Zephyr units remain after shipping?\'\n                    gold = a * b - c\n                distractors = [f\'Cedar{i} inventory: {rng.randint(100, 999)} units; \'\n                               f\'Cedar{i} shipment: {rng.randint(10, 99)} units.\' for i in range(24)]\n                positions = (\'clean\', \'middle\') if split == \'development\' else (\'clean\', \'early\', \'middle\', \'late\')\n                for position in positions:\n                    if position == \'clean\':\n                        records = facts[:]\n                    else:\n                        records = distractors[:]\n                        where = {\'early\': 0, \'middle\': len(records) // 2, \'late\': len(records)}[position]\n                        records[where:where] = facts\n                    user = \'\\n\'.join(records) + \'\\n\\n\' + query + \'\\nReturn only the integer.\'\n                    messages = [{\'role\': \'system\', \'content\': SYSTEM}, {\'role\': \'user\', \'content\': user}]\n                    splits[split].append({\'case_id\': f\'{family}-{position}\', \'family_id\': family,\n                        \'split\': split, \'operation\': operation, \'position\': position,\n                        \'operands\': [a, b, c], \'answer\': str(gold), \'evidence_text\': facts,\n                        \'messages\': messages, \'messages_sha256\': canonical_hash(messages)})\n    return splits\n\n\ndef render_case(case, tokenizer, max_prompt_tokens=1536):\n    text = tokenizer.apply_chat_template(case[\'messages\'], tokenize=False,\n                                         add_generation_prompt=True, enable_thinking=False)\n    ids = tokenizer(text, add_special_tokens=False).input_ids\n    if len(ids) > max_prompt_tokens:\n        raise ValueError(f"{case[\'case_id\']}: {len(ids)} tokens exceeds {max_prompt_tokens}; do not truncate evidence")\n    return {**case, \'prompt\': text, \'ids\': ids,\n            \'prompt_sha256\': hashlib.sha256(text.encode()).hexdigest()}\n\n\ndef development_gate(cases, results):\n    """Predeclared baseline gate, never case selection or a held-out filter."""\n    if cases != build_protocol()[\'development\']:\n        raise ValueError(\'Gate requires the complete frozen development set\')\n    expected = {c[\'case_id\']: c for c in cases}\n    by_id = {}\n    scores = []\n    for row in results:\n        key = row[\'case_id\']\n        if key not in expected or key in by_id:\n            raise ValueError(\'Unknown or duplicate development result\')\n        if row[\'policy\'] != \'native_fp16\':\n            raise ValueError(\'Development gate accepts only native FP16 results\')\n        case = expected[key]\n        if row[\'messages_sha256\'] != case[\'messages_sha256\']:\n            raise ValueError(\'Result prompt identity mismatch\')\n        scored = score_response(row[\'text\'], case[\'answer\'], ended_eos=row[\'ended_eos\'], truncated=row[\'truncated\'])\n        by_id[key] = scored\n        scores.append({**scored, \'case_id\': key, \'operation\': case[\'operation\'], \'position\': case[\'position\']})\n    correct = sum(s[\'final_completed_correct\'] for s in scores)\n    operation_counts = {op: sum(s[\'final_completed_correct\'] for s in scores if s[\'operation\'] == op) for op in OPERATIONS}\n    position_counts = {p: sum(s[\'final_completed_correct\'] for s in scores if s[\'position\'] == p) for p in (\'clean\', \'middle\')}\n    complete = len(results) == len(cases)\n    passed = complete and correct >= 11 and min(operation_counts.values()) >= 3 and min(position_counts.values()) >= 5\n    return {\'protocol_version\': PROTOCOL_VERSION, \'scorer_version\': SCORER_VERSION,\n            \'status\': \'incomplete\' if not complete else \'passed\' if passed else \'failed\',\n            \'correct\': correct, \'received\': len(results), \'expected\': len(cases),\n            \'correct_by_operation\': operation_counts, \'correct_by_position\': position_counts,\n            \'scores\': scores, \'warning\': \'Development readiness only; no held-out or compression claim. Never filter held-out cases using this gate.\'}\n\n\ndef export_protocol(directory):\n    directory = Path(directory)\n    if directory.exists():\n        raise ValueError(\'Choose a new directory; frozen artifacts are not overwritten\')\n    splits = build_protocol()\n    manifest = {\'protocol_version\': PROTOCOL_VERSION, \'scorer_version\': SCORER_VERSION,\n        \'model_id\': MODEL_ID, \'model_revision\': MODEL_REVISION,\n        \'generation\': {\'max_new_tokens\': 64, \'do_sample\': False, \'enable_thinking\': False},\n        \'max_prompt_tokens\': 1536, \'development_gate\': {\n            \'minimum_correct\': 11, \'total\': 12, \'minimum_correct_per_operation\': 3,\n            \'minimum_correct_per_context\': 5},\n        \'splits\': {name: {\'cases\': len(cases), \'families\': len({c[\'family_id\'] for c in cases}),\n                          \'sha256\': canonical_hash(cases)} for name, cases in splits.items()},\n        \'source_hash_encoding\': \'UTF-8 with universal newlines\',\n        \'source_sha256\': {name: hashlib.sha256((Path(__file__).parent / name).read_text(encoding=\'utf-8\').encode()).hexdigest()\n                          for name in (\'arithmetic_protocol.py\', \'arithmetic_scoring.py\')},\n        \'baseline_status\': \'not_run\', \'heldout_status\': \'not_evaluated\'}\n    directory.mkdir(parents=True)\n    for name, value in {**splits, \'manifest\': manifest}.items():\n        (directory / f\'{name}.json\').write_text(json.dumps(value, indent=2), encoding=\'utf-8\')\n    print(json.dumps(manifest, indent=2))\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'output_directory\')\n    export_protocol(parser.parse_args().output_directory)\n', 'arithmetic_scoring.py': '"""Frozen prospective arithmetic scoring v1; legacy post-hoc scoring is untouched."""\nimport re\n\nSCORER_VERSION = \'arithmetic-final-v1\'\nMAX_RESPONSE_CHARS = 20000\nINTEGER = r\'[+-]?[0-9]{1,12}\'\n\n\ndef extract_answer(text):\n    if not isinstance(text, str):\n        raise TypeError(\'response must be text\')\n    if len(text) > MAX_RESPONSE_CHARS:\n        return None, \'oversized\'\n    lines = [line.strip() for line in text.splitlines() if line.strip()]\n    if not lines:\n        return None, \'empty\'\n    final = lines[-1]\n    # Entire final line only. Never search preceding reasoning for the gold value.\n    rules = (\n        (\'bare_integer\', rf\'({INTEGER})[.]?\'),\n        (\'answer_label\', rf\'(?:final answer|answer|the integer is|return only the integer)\\s*:\\s*({INTEGER})[.]?\'),\n        (\'remaining_units\', rf\'({INTEGER}) units remain(?: in the Zephyr shipment)?[.]?\'),\n        # A syntactically complete integer expression; !=, <=, decimals, prose,\n        # alternatives, multiple equalities and unfinished arithmetic are rejected.\n        (\'terminal_equation\', rf\'{INTEGER}(?:\\s*[-+*/×]\\s*{INTEGER})+\\s*=\\s*({INTEGER})[.]?\'),\n    )\n    for rule, pattern in rules:\n        match = re.fullmatch(pattern, final, flags=re.IGNORECASE | re.ASCII)\n        if match:\n            return str(int(match.group(1))), rule\n    return None, \'unparsed\'\n\n\ndef score_response(text, gold, *, ended_eos, truncated):\n    if not isinstance(gold, str) or not re.fullmatch(INTEGER, gold, re.ASCII):\n        raise ValueError(\'gold must be an ASCII integer string\')\n    if type(ended_eos) is not bool or type(truncated) is not bool:\n        raise TypeError(\'completion flags must be booleans\')\n    extracted, rule = extract_answer(text)\n    complete = ended_eos and not truncated\n    matched = extracted == str(int(gold)) if extracted is not None else False\n    return {\n        \'scorer_version\': SCORER_VERSION,\n        \'strict_text_correct\': text.strip() == gold,\n        \'strict_completed_correct\': complete and text.strip() == gold,\n        \'extracted_answer\': extracted, \'extraction_rule\': rule,\n        \'parsed\': extracted is not None, \'complete\': complete,\n        \'final_completed_correct\': complete and matched,\n        \'status\': (\'incomplete\' if not complete else \'unparsed\' if extracted is None\n                   else \'correct_final_claim\' if matched else \'incorrect_final_claim\'),\n    }\n', 'run_control.py': '"""Atomic checkpoints and cooperative wall-time limits for notebook phases."""\nimport hashlib\nimport json\nimport time\nfrom pathlib import Path\n\n\nclass RunControl:\n    def __init__(self, directory, manifest, clock=time.perf_counter):\n        self.directory = Path(directory)\n        self.directory.mkdir(parents=True, exist_ok=True)\n        self.clock = clock\n        digest = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()\n        path = self.directory / \'resume-manifest.json\'\n        if path.exists() and json.loads(path.read_text())[\'sha256\'] != digest:\n            raise ValueError(\'Checkpoint configuration/source/environment mismatch. Use a new output directory.\')\n        self.save(\'resume-manifest.json\', {\'sha256\': digest, \'manifest\': manifest})\n\n    def load(self, filename):\n        path = self.directory / filename\n        return json.loads(path.read_text()) if path.exists() else []\n\n    def save(self, filename, value):\n        path = self.directory / filename\n        temporary = path.with_suffix(path.suffix + \'.tmp\')\n        temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding=\'utf-8\')\n        temporary.replace(path)\n\n    def start(self, phase, minutes):\n        if minutes <= 0:\n            raise ValueError(\'Phase time limit must be positive\')\n        self.phase = phase\n        self.started = self.clock()\n        self.deadline = self.started + minutes * 60\n        self.paused = False\n\n    def allow(self):\n        if self.clock() < self.deadline:\n            return True\n        if not self.paused:\n            print(f\'{self.phase}: time budget reached. Saved work retained; rerun to continue.\', flush=True)\n            self.save(f\'{self.phase}-status.json\', {\'status\': \'paused_time_budget\'})\n        self.paused = True\n        return False\n\n    def log(self, message):\n        elapsed = (self.clock() - self.started) / 60\n        print(f\'[{self.phase} {elapsed:.1f} min] {message}\', flush=True)\n'}
DEVELOPMENT_CASES = [{'case_id': 'development-addition-00-clean', 'family_id': 'development-addition-00', 'split': 'development', 'operation': 'addition', 'position': 'clean', 'operands': [75, 27, 0], 'answer': '102', 'evidence_text': ['Zephyr opening inventory: 75 units.', 'Zephyr received: 27 units.'], 'messages': [{'role': 'system', 'content': 'Solve the requested inventory calculation using only the Zephyr records. Other projects are unrelated. Return only the final integer, without units or explanation.'}, {'role': 'user', 'content': 'Zephyr opening inventory: 75 units.\nZephyr received: 27 units.\n\nWhat is Zephyr inventory after receiving the delivery?\nReturn only the integer.'}], 'messages_sha256': '894e41be36009fae1c2cb13bbbbf3d3a952b1300ce8f5e7b98f53038447f9a64'}, {'case_id': 'development-addition-00-middle', 'family_id': 'development-addition-00', 'split': 'development', 'operation': 'addition', 'position': 'middle', 'operands': [75, 27, 0], 'answer': '102', 'evidence_text': ['Zephyr opening inventory: 75 units.', 'Zephyr received: 27 units.'], 'messages': [{'role': 'system', 'content': 'Solve the requested inventory calculation using only the Zephyr records. Other projects are unrelated. Return only the final integer, without units or explanation.'}, {'role': 'user', 'content': 'Cedar0 inventory: 786 units; Cedar0 shipment: 66 units.\nCedar1 inventory: 603 units; Cedar1 shipment: 14 units.\nCedar2 inventory: 827 units; Cedar2 shipment: 23 units.\nCedar3 inventory: 622 units; Cedar3 shipment: 40 units.\nCedar4 inventory: 915 units; Cedar4 shipment: 37 units.\nCedar5 inventory: 639 units; Cedar5 shipment: 10 units.\nCedar6 inventory: 482 units; Cedar6 shipment: 83 units.\nCedar7 inventory: 236 units; Cedar7 shipment: 93 units.\nCedar8 inventory: 591 units; Cedar8 shipment: 83 units.\nCedar9 inventory: 699 units; Cedar9 shipment: 32 units.\nCedar10 inventory: 230 units; Cedar10 shipment: 13 units.\nCedar11 inventory: 726 units; Cedar11 shipment: 92 units.\nZephyr opening inventory: 75 units.\nZephyr received: 27 units.\nCedar12 inventory: 784 units; Cedar12 shipment: 69 units.\nCedar13 inventory: 606 units; Cedar13 shipment: 72 units.\nCedar14 inventory: 565 units; Cedar14 shipment: 10 units.\nCedar15 inventory: 222 units; Cedar15 shipment: 52 units.\nCedar16 inventory: 279 units; Cedar16 shipment: 84 units.\nCedar17 inventory: 405 units; Cedar17 shipment: 73 units.\nCedar18 inventory: 940 units; Cedar18 shipment: 91 units.\nCedar19 inventory: 797 units; Cedar19 shipment: 72 units.\nCedar20 inventory: 126 units; Cedar20 shipment: 97 units.\nCedar21 inventory: 264 units; Cedar21 shipment: 91 units.\nCedar22 inventory: 562 units; Cedar22 shipment: 77 units.\nCedar23 inventory: 587 units; Cedar23 shipment: 58 units.\n\nWhat is Zephyr inventory after receiving the delivery?\nReturn only the integer.'}], 'messages_sha256': '63c2c8599cf0eff1971ac5b0ecb2619f8593c0364aba019cd12335d300b7da3b'}, {'case_id': 'development-addition-01-clean', 'family_id': 'development-addition-01', 'split': 'development', 'operation': 'addition', 'position': 'clean', 'operands': [49, 14, 0], 'answer': '63', 'evidence_text': ['Zephyr opening inventory: 49 units.', 'Zephyr received: 14 units.'], 'messages': [{'role': 'system', 'content': 'Solve the requested inventory calculation using only the Zephyr records. Other projects are unrelated. Return only the final integer, without units or explanation.'}, {'role': 'user', 'content': 'Zephyr opening inventory: 49 units.\nZephyr received: 14 units.\n\nWhat is Zephyr inventory after receiving the delivery?\nReturn only the integer.'}], 'messages_sha256': '39d6eb7d819febdb747895c5285f84fd43babe70c5112e961016a5f528634038'}, {'case_id': 'development-addition-01-middle', 'family_id': 'development-addition-01', 'split': 'development', 'operation': 'addition', 'position': 'middle', 'operands': [49, 14, 0], 'answer': '63', 'evidence_text': ['Zephyr opening inventory: 49 units.', 'Zephyr received: 14 units.'], 'messages': [{'role': 'system', 'content': 'Solve the requested inventory calculation using only the Zephyr records. Other projects are unrelated. Return only the final integer, without units or explanation.'}, {'role': 'user', 'content': 'Cedar0 inventory: 280 units; Cedar0 shipment: 12 units.\nCedar1 inventory: 875 units; Cedar1 shipment: 60 units.\nCedar2 inventory: 201 units; Cedar2 shipment: 70 units.\nCedar3 inventory: 310 units; Cedar3 shipment: 67 units.\nCedar4 inventory: 997 units; Cedar4 shipment: 88 units.\nCedar5 inventory: 100 units; Cedar5 shipment: 37 units.\nCedar6 inventory: 968 units; Cedar6 shipment: 39 units.\nCedar7 inventory: 913 units; Cedar7 shipment: 86 units.\nCedar8 inventory: 589 units; Cedar8 shipment: 67 units.\nCedar9 inventory: 414 units; Cedar9 shipment: 67 units.\nCedar10 inventory: 608 units; Cedar10 shipment: 82 units.\nCedar11 inventory: 629 units; Cedar11 shipment: 67 units.\nZephyr opening inventory: 49 units.\nZephyr received: 14 units.\nCedar12 inventory: 835 units; Cedar12 shipment: 25 units.\nCedar13 inventory: 520 units; Cedar13 shipment: 18 units.\nCedar14 inventory: 661 units; Cedar14 shipment: 46 units.\nCedar15 inventory: 504 units; Cedar15 shipment: 84 units.\nCedar16 inventory: 840 units; Cedar16 shipment: 41 units.\nCedar17 inventory: 596 units; Cedar17 shipment: 71 units.\nCedar18 inventory: 950 units; Cedar18 shipment: 23 units.\nCedar19 inventory: 169 units; Cedar19 shipment: 46 units.\nCedar20 inventory: 812 units; Cedar20 shipment: 96 units.\nCedar21 inventory: 164 units; Cedar21 shipment: 12 units.\nCedar22 inventory: 231 units; Cedar22 shipment: 17 units.\nCedar23 inventory: 792 units; Cedar23 shipment: 64 units.\n\nWhat is Zephyr inventory after receiving the delivery?\nReturn only the integer.'}], 'messages_sha256': '000cd758fd8a1c9ad361cd814291b431174d6d07e42e1a408a20854c76eb460d'}, {'case_id': 'development-subtraction-00-clean', 'family_id': 'development-subtraction-00', 'split': 'development', 'operation': 'subtraction', 'position': 'clean', 'operands': [83, 13, 0], 'answer': '70', 'evidence_text': ['Zephyr opening inventory: 83 units.', 'Zephyr shipped: 13 units.'], 'messages': [{'role': 'system', 'content': 'Solve the requested inventory calculation using only the Zephyr records. Other projects are unrelated. Return only the final integer, without units or explanation.'}, {'role': 'user', 'content': 'Zephyr opening inventory: 83 units.\nZephyr shipped: 13 units.\n\nWhat is Zephyr inventory after shipping?\nReturn only the integer.'}], 'messages_sha256': '8aa76f4115e08daa064b84a04710f5bf2fc909580da1903072b0e12b638624f3'}, {'case_id': 'development-subtraction-00-middle', 'family_id': 'development-subtraction-00', 'split': 'development', 'operation': 'subtraction', 'position': 'middle', 'operands': [83, 13, 0], 'answer': '70', 'evidence_text': ['Zephyr opening inventory: 83 units.', 'Zephyr shipped: 13 units.'], 'messages': [{'role': 'system', 'content': 'Solve the requested inventory calculation using only the Zephyr records. Other projects are unrelated. Return only the final integer, without units or explanation.'}, {'role': 'user', 'content': 'Cedar0 inventory: 179 units; Cedar0 shipment: 89 units.\nCedar1 inventory: 702 units; Cedar1 shipment: 16 units.\nCedar2 inventory: 314 units; Cedar2 shipment: 72 units.\nCedar3 inventory: 552 units; Cedar3 shipment: 61 units.\nCedar4 inventory: 933 units; Cedar4 shipment: 34 units.\nCedar5 inventory: 840 units; Cedar5 shipment: 80 units.\nCedar6 inventory: 328 units; Cedar6 shipment: 16 units.\nCedar7 inventory: 343 units; Cedar7 shipment: 26 units.\nCedar8 inventory: 741 units; Cedar8 shipment: 26 units.\nCedar9 inventory: 110 units; Cedar9 shipment: 97 units.\nCedar10 inventory: 765 units; Cedar10 shipment: 41 units.\nCedar11 inventory: 328 units; Cedar11 shipment: 36 units.\nZephyr opening inventory: 83 units.\nZephyr shipped: 13 units.\nCedar12 inventory: 127 units; Cedar12 shipment: 47 units.\nCedar13 inventory: 957 units; Cedar13 shipment: 66 units.\nCedar14 inventory: 102 units; Cedar14 shipment: 14 units.\nCedar15 inventory: 417 units; Cedar15 shipment: 62 units.\nCedar16 inventory: 912 units; Cedar16 shipment: 34 units.\nCedar17 inventory: 752 units; Cedar17 shipment: 19 units.\nCedar18 inventory: 944 units; Cedar18 shipment: 45 units.\nCedar19 inventory: 501 units; Cedar19 shipment: 63 units.\nCedar20 inventory: 487 units; Cedar20 shipment: 41 units.\nCedar21 inventory: 856 units; Cedar21 shipment: 42 units.\nCedar22 inventory: 428 units; Cedar22 shipment: 29 units.\nCedar23 inventory: 542 units; Cedar23 shipment: 95 units.\n\nWhat is Zephyr inventory after shipping?\nReturn only the integer.'}], 'messages_sha256': 'c4b1d5861e93c858075ba96a2fdd9775109ec88deaa34cd4ac03550ee08b44ef'}, {'case_id': 'development-subtraction-01-clean', 'family_id': 'development-subtraction-01', 'split': 'development', 'operation': 'subtraction', 'position': 'clean', 'operands': [68, 8, 0], 'answer': '60', 'evidence_text': ['Zephyr opening inventory: 68 units.', 'Zephyr shipped: 8 units.'], 'messages': [{'role': 'system', 'content': 'Solve the requested inventory calculation using only the Zephyr records. Other projects are unrelated. Return only the final integer, without units or explanation.'}, {'role': 'user', 'content': 'Zephyr opening inventory: 68 units.\nZephyr shipped: 8 units.\n\nWhat is Zephyr inventory after shipping?\nReturn only the integer.'}], 'messages_sha256': '0f8958c327a8f276cb8185240f14844ff5ed7fd5dd55f7242bd360b811337528'}, {'case_id': 'development-subtraction-01-middle', 'family_id': 'development-subtraction-01', 'split': 'development', 'operation': 'subtraction', 'position': 'middle', 'operands': [68, 8, 0], 'answer': '60', 'evidence_text': ['Zephyr opening inventory: 68 units.', 'Zephyr shipped: 8 units.'], 'messages': [{'role': 'system', 'content': 'Solve the requested inventory calculation using only the Zephyr records. Other projects are unrelated. Return only the final integer, without units or explanation.'}, {'role': 'user', 'content': 'Cedar0 inventory: 426 units; Cedar0 shipment: 23 units.\nCedar1 inventory: 883 units; Cedar1 shipment: 28 units.\nCedar2 inventory: 804 units; Cedar2 shipment: 33 units.\nCedar3 inventory: 599 units; Cedar3 shipment: 52 units.\nCedar4 inventory: 965 units; Cedar4 shipment: 94 units.\nCedar5 inventory: 613 units; Cedar5 shipment: 19 units.\nCedar6 inventory: 215 units; Cedar6 shipment: 38 units.\nCedar7 inventory: 556 units; Cedar7 shipment: 10 units.\nCedar8 inventory: 458 units; Cedar8 shipment: 13 units.\nCedar9 inventory: 236 units; Cedar9 shipment: 56 units.\nCedar10 inventory: 921 units; Cedar10 shipment: 37 units.\nCedar11 inventory: 628 units; Cedar11 shipment: 49 units.\nZephyr opening inventory: 68 units.\nZephyr shipped: 8 units.\nCedar12 inventory: 648 units; Cedar12 shipment: 50 units.\nCedar13 inventory: 976 units; Cedar13 shipment: 24 units.\nCedar14 inventory: 429 units; Cedar14 shipment: 41 units.\nCedar15 inventory: 641 units; Cedar15 shipment: 26 units.\nCedar16 inventory: 328 units; Cedar16 shipment: 59 units.\nCedar17 inventory: 130 units; Cedar17 shipment: 87 units.\nCedar18 inventory: 302 units; Cedar18 shipment: 46 units.\nCedar19 inventory: 389 units; Cedar19 shipment: 46 units.\nCedar20 inventory: 617 units; Cedar20 shipment: 92 units.\nCedar21 inventory: 412 units; Cedar21 shipment: 98 units.\nCedar22 inventory: 987 units; Cedar22 shipment: 19 units.\nCedar23 inventory: 593 units; Cedar23 shipment: 19 units.\n\nWhat is Zephyr inventory after shipping?\nReturn only the integer.'}], 'messages_sha256': 'f7bc6caeacb9b1de2be1dcd8a1877e32f1197b09d85e742c11f544939fbd600c'}, {'case_id': 'development-multiply_subtract-00-clean', 'family_id': 'development-multiply_subtract-00', 'split': 'development', 'operation': 'multiply_subtract', 'position': 'clean', 'operands': [7, 5, 9], 'answer': '26', 'evidence_text': ['Zephyr received: 7 crates.', 'Zephyr units per crate: 5.', 'Zephyr shipped after receiving: 9 units.'], 'messages': [{'role': 'system', 'content': 'Solve the requested inventory calculation using only the Zephyr records. Other projects are unrelated. Return only the final integer, without units or explanation.'}, {'role': 'user', 'content': 'Zephyr received: 7 crates.\nZephyr units per crate: 5.\nZephyr shipped after receiving: 9 units.\n\nHow many Zephyr units remain after shipping?\nReturn only the integer.'}], 'messages_sha256': '80db0dc37ee6d31fc9a5d32c025943a56bd27bf2d89156986698019960317745'}, {'case_id': 'development-multiply_subtract-00-middle', 'family_id': 'development-multiply_subtract-00', 'split': 'development', 'operation': 'multiply_subtract', 'position': 'middle', 'operands': [7, 5, 9], 'answer': '26', 'evidence_text': ['Zephyr received: 7 crates.', 'Zephyr units per crate: 5.', 'Zephyr shipped after receiving: 9 units.'], 'messages': [{'role': 'system', 'content': 'Solve the requested inventory calculation using only the Zephyr records. Other projects are unrelated. Return only the final integer, without units or explanation.'}, {'role': 'user', 'content': 'Cedar0 inventory: 760 units; Cedar0 shipment: 48 units.\nCedar1 inventory: 676 units; Cedar1 shipment: 13 units.\nCedar2 inventory: 327 units; Cedar2 shipment: 89 units.\nCedar3 inventory: 815 units; Cedar3 shipment: 16 units.\nCedar4 inventory: 873 units; Cedar4 shipment: 77 units.\nCedar5 inventory: 474 units; Cedar5 shipment: 25 units.\nCedar6 inventory: 910 units; Cedar6 shipment: 11 units.\nCedar7 inventory: 730 units; Cedar7 shipment: 70 units.\nCedar8 inventory: 834 units; Cedar8 shipment: 42 units.\nCedar9 inventory: 825 units; Cedar9 shipment: 39 units.\nCedar10 inventory: 815 units; Cedar10 shipment: 86 units.\nCedar11 inventory: 522 units; Cedar11 shipment: 47 units.\nZephyr received: 7 crates.\nZephyr units per crate: 5.\nZephyr shipped after receiving: 9 units.\nCedar12 inventory: 633 units; Cedar12 shipment: 29 units.\nCedar13 inventory: 974 units; Cedar13 shipment: 87 units.\nCedar14 inventory: 921 units; Cedar14 shipment: 55 units.\nCedar15 inventory: 477 units; Cedar15 shipment: 14 units.\nCedar16 inventory: 454 units; Cedar16 shipment: 35 units.\nCedar17 inventory: 187 units; Cedar17 shipment: 57 units.\nCedar18 inventory: 427 units; Cedar18 shipment: 23 units.\nCedar19 inventory: 296 units; Cedar19 shipment: 27 units.\nCedar20 inventory: 478 units; Cedar20 shipment: 46 units.\nCedar21 inventory: 860 units; Cedar21 shipment: 80 units.\nCedar22 inventory: 706 units; Cedar22 shipment: 36 units.\nCedar23 inventory: 441 units; Cedar23 shipment: 41 units.\n\nHow many Zephyr units remain after shipping?\nReturn only the integer.'}], 'messages_sha256': '32c655bddac490d047bf50194ad5b4525336e55aee87cede9f08781afe7431db'}, {'case_id': 'development-multiply_subtract-01-clean', 'family_id': 'development-multiply_subtract-01', 'split': 'development', 'operation': 'multiply_subtract', 'position': 'clean', 'operands': [10, 10, 9], 'answer': '91', 'evidence_text': ['Zephyr received: 10 crates.', 'Zephyr units per crate: 10.', 'Zephyr shipped after receiving: 9 units.'], 'messages': [{'role': 'system', 'content': 'Solve the requested inventory calculation using only the Zephyr records. Other projects are unrelated. Return only the final integer, without units or explanation.'}, {'role': 'user', 'content': 'Zephyr received: 10 crates.\nZephyr units per crate: 10.\nZephyr shipped after receiving: 9 units.\n\nHow many Zephyr units remain after shipping?\nReturn only the integer.'}], 'messages_sha256': '5279a68b6f5a72d15b30163c0a6e8e817486ba7ca762d43bf049384f2abed758'}, {'case_id': 'development-multiply_subtract-01-middle', 'family_id': 'development-multiply_subtract-01', 'split': 'development', 'operation': 'multiply_subtract', 'position': 'middle', 'operands': [10, 10, 9], 'answer': '91', 'evidence_text': ['Zephyr received: 10 crates.', 'Zephyr units per crate: 10.', 'Zephyr shipped after receiving: 9 units.'], 'messages': [{'role': 'system', 'content': 'Solve the requested inventory calculation using only the Zephyr records. Other projects are unrelated. Return only the final integer, without units or explanation.'}, {'role': 'user', 'content': 'Cedar0 inventory: 221 units; Cedar0 shipment: 82 units.\nCedar1 inventory: 403 units; Cedar1 shipment: 69 units.\nCedar2 inventory: 356 units; Cedar2 shipment: 18 units.\nCedar3 inventory: 672 units; Cedar3 shipment: 31 units.\nCedar4 inventory: 175 units; Cedar4 shipment: 34 units.\nCedar5 inventory: 808 units; Cedar5 shipment: 19 units.\nCedar6 inventory: 318 units; Cedar6 shipment: 45 units.\nCedar7 inventory: 702 units; Cedar7 shipment: 46 units.\nCedar8 inventory: 880 units; Cedar8 shipment: 57 units.\nCedar9 inventory: 794 units; Cedar9 shipment: 96 units.\nCedar10 inventory: 267 units; Cedar10 shipment: 54 units.\nCedar11 inventory: 345 units; Cedar11 shipment: 14 units.\nZephyr received: 10 crates.\nZephyr units per crate: 10.\nZephyr shipped after receiving: 9 units.\nCedar12 inventory: 854 units; Cedar12 shipment: 69 units.\nCedar13 inventory: 960 units; Cedar13 shipment: 40 units.\nCedar14 inventory: 791 units; Cedar14 shipment: 18 units.\nCedar15 inventory: 552 units; Cedar15 shipment: 11 units.\nCedar16 inventory: 832 units; Cedar16 shipment: 20 units.\nCedar17 inventory: 859 units; Cedar17 shipment: 66 units.\nCedar18 inventory: 359 units; Cedar18 shipment: 96 units.\nCedar19 inventory: 285 units; Cedar19 shipment: 17 units.\nCedar20 inventory: 999 units; Cedar20 shipment: 12 units.\nCedar21 inventory: 599 units; Cedar21 shipment: 93 units.\nCedar22 inventory: 747 units; Cedar22 shipment: 64 units.\nCedar23 inventory: 873 units; Cedar23 shipment: 87 units.\n\nHow many Zephyr units remain after shipping?\nReturn only the integer.'}], 'messages_sha256': '7ad10e877f0b745bab7fa4823d96736fd23d57c23ea7ce0fdab91757fe635db7'}]
SOURCE_SHA256 = '18efd3b4b2a749515d3cc5b9489666fad45581db6d3743d5664212554be88aff'
WORKFLOW_SHA256 = '237540383fa208210def76b46990524bb51cf2bb65e4c06f1aade8cec8d8d9fa'
runtime = Path('/content/anchorkv-baseline-runtime')
package = runtime / 'anchorkv_baseline'
package.mkdir(parents=True, exist_ok=True)
for name, source in SOURCES.items():
    (package / name).write_text(source, encoding='utf-8')
sys.path.insert(0, str(runtime))

In [ ]:
from anchorkv_baseline.arithmetic_protocol import (
    MODEL_ID, MODEL_REVISION, SCORER_VERSION, PROTOCOL_VERSION,
    canonical_hash, render_case, development_gate,
)
from anchorkv_baseline.run_control import RunControl
RESUME_DIRECTORY = None
PHASE_MINUTES = 8
OUTPUT = Path(RESUME_DIRECTORY) if RESUME_DIRECTORY else Path('/content/anchorkv-fp16-development') / time.strftime('%Y%m%d-%H%M%S')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
cases = [render_case(c, tokenizer) for c in DEVELOPMENT_CASES]
environment = {'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'torch': str(torch.__version__),
               'transformers': transformers.__version__, 'python': platform.python_version(),
               'gpu': torch.cuda.get_device_name(0), 'cuda': torch.version.cuda}
if RESUME_DIRECTORY and not (OUTPUT / 'resume-manifest.json').exists():
    raise ValueError('Resume directory lacks a compatible manifest; start a new run.')
checkpoint = RunControl(OUTPUT, {'environment': environment, 'source_sha256': SOURCE_SHA256,
    'workflow_sha256': WORKFLOW_SHA256, 'cases_sha256': canonical_hash(DEVELOPMENT_CASES),
    'max_new_tokens': 64, 'do_sample': False, 'protocol_version': PROTOCOL_VERSION,
    'scorer_version': SCORER_VERSION, 'prompt_hashes': [c['prompt_sha256'] for c in cases]})
checkpoint.save('prompts.json', cases)
checkpoint.save('environment.json', environment)
print('12 native FP16 generations; no held-out cases or compressed policies.')
print('Output/resume folder:', OUTPUT)
for case in cases:
    print(case['case_id'], len(case['ids']), 'prompt tokens')

In [ ]:
torch.manual_seed(20260910)
torch.cuda.manual_seed_all(20260910)
torch.set_num_threads(2)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=MODEL_REVISION,
    torch_dtype=torch.float16, attn_implementation='sdpa', low_cpu_mem_usage=True).to('cuda').eval()

In [ ]:
rows = checkpoint.load('development-results.json')
# Validate loaded identities/completion records before trusting them as completed work.
development_gate(DEVELOPMENT_CASES, rows)
done = {r['case_id'] for r in rows}
checkpoint.start('development', PHASE_MINUTES)
stop_ids = model.generation_config.eos_token_id
stop_ids = set(stop_ids if isinstance(stop_ids, list) else [stop_ids]) | {tokenizer.eos_token_id}
stop_ids.discard(None)
for case in cases:
    if case['case_id'] in done:
        continue
    if not checkpoint.allow():
        break
    checkpoint.log(f"Generating {case['case_id']} ({len(done)}/12 saved)")
    input_ids = torch.tensor([case['ids']], dtype=torch.long, device=model.device)
    torch.cuda.synchronize()
    started = time.perf_counter()
    with torch.inference_mode():
        output = model.generate(input_ids=input_ids, attention_mask=torch.ones_like(input_ids),
            max_new_tokens=64, do_sample=False, use_cache=True,
            eos_token_id=sorted(stop_ids), pad_token_id=tokenizer.eos_token_id)
    torch.cuda.synchronize()
    seconds = time.perf_counter() - started
    tokens = output[0, input_ids.shape[1]:].tolist()
    ended = bool(tokens and tokens[-1] in stop_ids)
    rows.append({'case_id': case['case_id'], 'policy': 'native_fp16',
        'messages_sha256': case['messages_sha256'], 'prompt_sha256': case['prompt_sha256'],
        'text': tokenizer.decode(tokens, skip_special_tokens=True), 'generated_ids': tokens,
        'ended_eos': ended, 'truncated': not ended, 'generation_seconds': seconds})
    checkpoint.save('development-results.json', rows)
    done.add(case['case_id'])
    checkpoint.log(f'Saved {len(done)}/12')
    del input_ids, output
    torch.cuda.empty_cache()

In [ ]:
gate = development_gate(DEVELOPMENT_CASES, checkpoint.load('development-results.json'))
checkpoint.save('development-gate.json', gate)
print('Gate:', gate['status'], '| correct final answers:', gate['correct'], '/', gate['expected'])
print('By operation:', gate['correct_by_operation'])
print('By context:', gate['correct_by_position'])
print('Pass requires at least 11/12 overall, 3/4 per operation, and 5/6 per context.')
if gate['status'] == 'incomplete':
    print('Rerun generation, then this report and download cell. Completed cases are skipped.')
elif gate['status'] == 'failed':
    print('Do not run the held-out compression sweep yet. Return the failures for development review.')
else:
    print('Development readiness passed. Held-out performance and compression benefit remain untested.')
report = ['# FP16 development readiness', '', json.dumps(gate, indent=2), '',
          'Strict and final-answer scores are separate. Unparsed is not a proven arithmetic error.',
          'Matched contexts are not independent questions. This is six underlying problems.',
          'No held-out inference or compression comparison was performed.']
(OUTPUT / 'report.md').write_text('\n'.join(report), encoding='utf-8')

In [ ]:
source_dir = OUTPUT / 'runtime-source'
source_dir.mkdir(exist_ok=True)
for name, source in SOURCES.items():
    (source_dir / name).write_text(source, encoding='utf-8')
archive = shutil.make_archive(str(OUTPUT), 'zip', OUTPUT)
print('Results:', archive)
from google.colab import files
files.download(archive)